# Cybersecurity Synthetic Dataset — Regression Benchmarks (OLS vs Regularization)

This notebook uses the provided synthetic cybersecurity telemetry dataset to compare:
- **Linear Regression (OLS)**
- **Ridge Regression (L2)**
- **Lasso Regression (L1)**
- **Elastic Net (L1 + L2)**
- **SGDRegressor (elasticnet penalty)**
- **Huber Regressor (robust)**
- **Bayesian Ridge (probabilistic linear regression)**

It also includes **multi-color visualizations** for:
- Target distribution and class distribution
- PCA projection (colored by attack type)
- Model performance comparison (R² / RMSE)
- Predicted vs Actual and residual plots (colored by attack type)
- Coefficient importance plots (Ridge / Lasso)


In [ ]:
# --- Imports and config ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import (
    LinearRegression,
    RidgeCV,
    LassoCV,
    ElasticNetCV,
    SGDRegressor,
    HuberRegressor,
    BayesianRidge,
)

from sklearn.decomposition import PCA

# Reproducibility
RANDOM_STATE = 7
np.random.seed(RANDOM_STATE)

# Dataset path (adjust if needed)
DATA_PATH = "cyber_15000.csv"


In [ ]:
# --- Load dataset ---
df = pd.read_csv(DATA_PATH, low_memory=False)

# Parse timestamp (kept for feature engineering)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], errors="coerce")

print("Shape:", df.shape)
display(df.head(3))


In [ ]:
# --- Quick data audit ---
targets = ["incident_impact_score", "attack_type"]
missing = df.isna().mean().sort_values(ascending=False)

print("Targets:", targets)
print("\nMissing values (top 15):")
display(missing.head(15))

print("\nAttack type counts:")
display(df["attack_type"].value_counts())


## 1) Exploratory Visualizations (multi-color)

We visualize:
- distribution of the regression target (`incident_impact_score`)
- class distribution (`attack_type`)


In [ ]:
# --- Target distribution & class distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Target histogram
axes[0].hist(df["incident_impact_score"].values, bins=40, edgecolor="white")
axes[0].set_title("Incident Impact Score Distribution")
axes[0].set_xlabel("incident_impact_score")
axes[0].set_ylabel("count")

# Class distribution (multi-color bars)
class_counts = df["attack_type"].value_counts()
classes = class_counts.index.tolist()
counts = class_counts.values

cmap = plt.cm.get_cmap("tab10", len(classes))
bar_colors = [cmap(i) for i in range(len(classes))]

axes[1].bar(classes, counts, color=bar_colors, edgecolor="white")
axes[1].set_title("Attack Type Distribution")
axes[1].set_xlabel("attack_type")
axes[1].set_ylabel("count")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


## 2) Feature Engineering

We keep the dataset largely "as-is", but add a few lightweight time features:
- hour of day
- day of week

We also:
- drop `record_id` (identifier)
- convert boolean columns to 0/1 integers


In [ ]:
# --- Feature engineering ---
df_fe = df.copy()

# Add time-derived features
df_fe["hour_utc"] = df_fe["timestamp_utc"].dt.hour.astype("Int64")
df_fe["dayofweek_utc"] = df_fe["timestamp_utc"].dt.dayofweek.astype("Int64")

# Fill missing time values with median (defensive)
for c in ["hour_utc", "dayofweek_utc"]:
    df_fe[c] = df_fe[c].fillna(int(df_fe[c].median()))

# Convert booleans -> ints (0/1)
bool_cols = df_fe.select_dtypes(include=["bool"]).columns.tolist()
for c in bool_cols:
    df_fe[c] = df_fe[c].astype(np.int8)

# Drop raw datetime and ID
df_fe = df_fe.drop(columns=["timestamp_utc", "record_id"])

# Define X and y
y = df_fe["incident_impact_score"].astype(np.float32)
attack_type = df_fe["attack_type"].astype(str)  # used for coloring / stratified split
X = df_fe.drop(columns=["incident_impact_score", "attack_type"])

print("X shape:", X.shape)
print("y shape:", y.shape)
display(X.head(3))


## 3) Train/Test Split

We **stratify on `attack_type`** so the train/test sets have similar proportions of attack categories.


In [ ]:
X_train, X_test, y_train, y_test, at_train, at_test = train_test_split(
    X, y, attack_type,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=attack_type
)

print("Train:", X_train.shape, " Test:", X_test.shape)


## 4) PCA Visualization (colored by attack type)

A 2D PCA projection helps you *see* dataset structure + overlap between classes.


In [ ]:
# --- Standardize (fit on training to avoid leakage) ---
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# --- PCA (2D) ---
pca = PCA(n_components=2, random_state=RANDOM_STATE)
Z_train = pca.fit_transform(X_train_s)

# Map classes -> colors
classes = sorted(at_train.unique())
cmap = plt.cm.get_cmap("tab10", len(classes))
color_map = {c: cmap(i) for i, c in enumerate(classes)}

plt.figure(figsize=(10, 7))
for c in classes:
    idx = (at_train.values == c)
    plt.scatter(Z_train[idx, 0], Z_train[idx, 1], s=10, alpha=0.55, color=color_map[c], label=c)

plt.title("PCA (2D) of Standardized Features (Train Set)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(markerscale=2, frameon=True)
plt.tight_layout()
plt.show()

print("Explained variance ratio:", pca.explained_variance_ratio_)


## 5) Model Training & Evaluation

All models are trained on **standardized features** (important for Lasso/ElasticNet/SGD).

Metrics:
- **R²** (higher is better)
- **RMSE** (lower is better)
- **MAE** (lower is better)


In [ ]:
def evaluate_model(name, model, X_train_s, X_test_s, y_train, y_test):
    """Fit a model and return metrics + predictions."""
    model.fit(X_train_s, y_train)
    pred = model.predict(X_test_s)

    rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
    mae = float(mean_absolute_error(y_test, pred))
    r2 = float(r2_score(y_test, pred))

    return {
        "model": name,
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "pred": pred,
        "estimator": model,
    }

# --- Define regression models ---
alphas_ridge = np.logspace(-3, 3, 40)
alphas_lasso = np.logspace(-4, 1, 50)

models = [
    ("LinearRegression (OLS)", LinearRegression()),
    ("RidgeCV (L2)", RidgeCV(alphas=alphas_ridge, cv=5)),
    ("LassoCV (L1)", LassoCV(alphas=alphas_lasso, cv=5, max_iter=5000, random_state=RANDOM_STATE)),
    ("ElasticNetCV (L1+L2)", ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
        alphas=alphas_lasso,
        cv=5,
        max_iter=5000,
        random_state=RANDOM_STATE
    )),
    ("SGDRegressor (elasticnet)", SGDRegressor(
        loss="squared_error",
        penalty="elasticnet",
        alpha=1e-4,
        l1_ratio=0.15,
        max_iter=5000,
        tol=1e-4,
        random_state=RANDOM_STATE
    )),
    ("HuberRegressor (robust)", HuberRegressor(epsilon=1.35, max_iter=500)),
    ("BayesianRidge", BayesianRidge()),
]

results = []
for name, m in models:
    results.append(evaluate_model(name, m, X_train_s, X_test_s, y_train, y_test))

res_df = pd.DataFrame([{k: v for k, v in r.items() if k in ["model", "R2", "RMSE", "MAE"]} for r in results])
res_df = res_df.sort_values(by="R2", ascending=False).reset_index(drop=True)

display(res_df)


## 6) Performance Comparison (multi-color)

Horizontal bar charts for R² and RMSE with distinct colors per model.


In [ ]:
models_order = res_df["model"].tolist()
r2_vals = res_df["R2"].values
rmse_vals = res_df["RMSE"].values

cmap = plt.cm.get_cmap("tab20", len(models_order))
colors = [cmap(i) for i in range(len(models_order))]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(models_order, r2_vals, color=colors, edgecolor="white")
axes[0].set_title("R² by Model (higher is better)")
axes[0].set_xlabel("R²")

axes[1].barh(models_order, rmse_vals, color=colors, edgecolor="white")
axes[1].set_title("RMSE by Model (lower is better)")
axes[1].set_xlabel("RMSE")

plt.tight_layout()
plt.show()


## 7) Best Model Diagnostics (colored by attack type)

We select the best model by **highest R²** and plot:
- Predicted vs Actual (color = attack_type)
- Residuals vs Predicted (color = attack_type)


In [ ]:
best_name = res_df.loc[0, "model"]
best = next(r for r in results if r["model"] == best_name)

best_pred = best["pred"]
print("Best model:", best_name)
print("R2:", r2_score(y_test, best_pred), "RMSE:", np.sqrt(mean_squared_error(y_test, best_pred)))

# Colors by attack type (test set)
classes_test = sorted(at_test.unique())
cmap = plt.cm.get_cmap("tab10", len(classes_test))
color_map = {c: cmap(i) for i, c in enumerate(classes_test)}
point_colors = [color_map[c] for c in at_test.values]

# Predicted vs Actual
plt.figure(figsize=(8, 7))
plt.scatter(y_test, best_pred, s=12, alpha=0.6, c=point_colors)
mn = float(min(y_test.min(), best_pred.min()))
mx = float(max(y_test.max(), best_pred.max()))
plt.plot([mn, mx], [mn, mx], linestyle="--", linewidth=2, color="black", alpha=0.7)
plt.title(f"Predicted vs Actual — {best_name}")
plt.xlabel("Actual incident_impact_score")
plt.ylabel("Predicted incident_impact_score")
plt.tight_layout()
plt.show()

# Residuals vs Predicted
residuals = y_test.values - best_pred
plt.figure(figsize=(8, 7))
plt.scatter(best_pred, residuals, s=12, alpha=0.6, c=point_colors)
plt.axhline(0, linestyle="--", linewidth=2, color="black", alpha=0.7)
plt.title(f"Residuals vs Predicted — {best_name}")
plt.xlabel("Predicted incident_impact_score")
plt.ylabel("Residual (actual - pred)")
plt.tight_layout()
plt.show()

# Legend for colors
plt.figure(figsize=(8, 1.2))
for c in classes_test:
    plt.scatter([], [], color=color_map[c], label=c, s=50)
plt.legend(ncol=3, frameon=True, loc="center")
plt.axis("off")
plt.show()


## 8) Coefficient Importance (Ridge vs Lasso)

We plot the **top 25 absolute coefficients** for Ridge and Lasso (if available).
Bars are colored by coefficient sign.


In [ ]:
feature_names = X.columns.to_list()

def plot_top_coeffs(estimator, title, top_k=25):
    if not hasattr(estimator, "coef_"):
        print(f"{title}: estimator has no coef_")
        return

    coef = np.asarray(estimator.coef_).ravel()
    abs_coef = np.abs(coef)

    top_idx = np.argsort(abs_coef)[-top_k:][::-1]
    top_features = [feature_names[i] for i in top_idx]
    top_values = coef[top_idx]

    # multi-color bars by sign (red-ish for negative, blue-ish for positive)
    colors = [plt.cm.RdBu(0.85) if v > 0 else plt.cm.RdBu(0.15) for v in top_values]

    plt.figure(figsize=(10, 7))
    plt.barh(top_features[::-1], top_values[::-1], color=colors[::-1], edgecolor="white")
    plt.title(title)
    plt.xlabel("Coefficient value (signed)")
    plt.tight_layout()
    plt.show()

ridge_est = next(r["estimator"] for r in results if r["model"].startswith("RidgeCV"))
lasso_est = next(r["estimator"] for r in results if r["model"].startswith("LassoCV"))

plot_top_coeffs(ridge_est, "Top Coefficients — RidgeCV (L2)", top_k=25)
plot_top_coeffs(lasso_est, "Top Coefficients — LassoCV (L1)", top_k=25)

if hasattr(ridge_est, "alpha_"):
    print("RidgeCV chosen alpha:", ridge_est.alpha_)
if hasattr(lasso_est, "alpha_"):
    print("LassoCV chosen alpha:", lasso_est.alpha_)


## 9) Correlation Heatmap (subset of features)

A heatmap on a **small subset** of features helps visualize multicollinearity.


In [ ]:
# Pick a manageable subset of columns for correlation visualization
subset_cols = []

preferred = [
    "bytes_out", "bytes_in", "unique_dst_ips", "unique_dst_ports",
    "dst_port_entropy", "tcp_syn_rate", "dns_query_rate",
    "failed_logins", "priv_escalation_events", "new_process_rate",
    "edr_alert_count", "siem_rule_hits", "threat_intel_hits",
    "dns_tunnel_score", "lateral_movement_score", "service_error_rate",
    "hour_utc", "dayofweek_utc"
]
subset_cols.extend([c for c in preferred if c in X.columns])

derived_cols = [c for c in X.columns if c.startswith("derived_")][:14]
subset_cols.extend(derived_cols)

subset_cols = list(dict.fromkeys(subset_cols))[:30]

corr = pd.DataFrame(X_train[subset_cols]).corr()

plt.figure(figsize=(10, 8))
plt.imshow(corr.values, aspect="auto", interpolation="nearest", cmap="coolwarm")
plt.colorbar(label="correlation")
plt.xticks(range(len(subset_cols)), subset_cols, rotation=90, fontsize=8)
plt.yticks(range(len(subset_cols)), subset_cols, fontsize=8)
plt.title("Correlation Heatmap (Selected Features)")
plt.tight_layout()
plt.show()


## 10) Next Steps (optional)

If you want even stronger performance while staying “linear-ish”, try:
- **Elastic Net with more `l1_ratio` values**
- **Feature selection (e.g., keep only top-|coef| from Ridge)**
- **Interactions** (polynomial features) + regularization (can be powerful but may overfit if not careful)
